# Train YOLOv8 trên VisDrone (bắt người/xe từ góc nhìn drone)

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**. Rồi Runtime → Run all.

- Dữ liệu VisDrone2019-DET (~2 GB) tự tải lần đầu.
- Kết quả + checkpoint lưu vào Google Drive (`MyDrive/uav_visdrone`) → Colab ngắt giữa chừng thì chạy lại notebook, nó **tự train tiếp** từ lần lưu cuối.
- Xong: tải file `visdrone_best.pt` về, chép vào `backend/models/visdrone_best.pt` rồi khởi động lại backend.

In [ ]:
!nvidia-smi -L   # phải thấy 'Tesla T4' (hoặc GPU khác); nếu lỗi -> chưa bật GPU
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/uav_visdrone'

# Chỉnh ở đây nếu cần
MODEL  = 'yolov8n.pt'  # 'yolov8s.pt' chính xác hơn nhưng chạy chậm hơn ~2-3x trên máy của anh
EPOCHS = 100           # T4: khoảng 2-3 phút/epoch với yolov8n -> ~4-5 giờ; giảm xuống 50 nếu vội
IMGSZ  = 640           # giữ 640 để khớp tốc độ nhận diện hiện tại trên máy

In [ ]:
import os
from ultralytics import YOLO

last = f'{PROJECT}/visdrone/weights/last.pt'
if os.path.exists(last):
    print('Tìm thấy checkpoint -> train tiếp:', last)
    model = YOLO(last)
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(data='VisDrone.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=32, device=0,
                project=PROJECT, name='visdrone', exist_ok=True, patience=20)

In [ ]:
# Đánh giá trên tập test-dev (có đáp án) và so với model COCO gốc đang dùng
best = f'{PROJECT}/visdrone/weights/best.pt'
new = YOLO(best).val(data='VisDrone.yaml', split='test', imgsz=IMGSZ, device=0,
                     project=PROJECT, name='val-test', exist_ok=True)
print(f'Model VisDrone  mAP50 = {new.box.map50:.3f}   mAP50-95 = {new.box.map:.3f}')

In [ ]:
import shutil
from google.colab import files
shutil.copy(best, '/content/visdrone_best.pt')
files.download('/content/visdrone_best.pt')   # chép file này vào backend/models/visdrone_best.pt